<a href="https://colab.research.google.com/github/simonburner/hands-on-LLM/blob/main/ch7/chains-n-memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langchain-community llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 27.6 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.34-py3-none-linux_x86_64.whl size=20581065 sha256=1db23c0e3a3e0103606cc1c13c333a64473dfa9d1f079e121e3d4d5a26e16346
  Stored in directory: /root/.cache/pip/wheels/4a/10/e7/0eb9b120f1640844f33562a3964c5b18b67de1d66d3f9530e8
Successfully built llama-cpp-python
  Attempting u

In [ ]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2026-07-28 15:44:18--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 13.226.251.20, 13.226.251.81, 13.226.251.112, ...
Connecting to huggingface.co (huggingface.co)|13.226.251.20|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/a9cdcf6e9514941ea9e596583b3d3c44dd99359fb7dd57f322bb84a0adc12ad4?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&user_id=public&Expires=1785257058&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2E5Y2RjZjZlOTUxNDk0MWVhOWU1OTY1ODNiM2QzYzQ0ZGQ5OTM1OWZiN2RkNTdmMzIyYmI4NGEwYWRjMTJhZDRcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitm

In [ ]:
from langchain_community.llms import LlamaCpp

/tmp/ipykernel_58/174005834.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import LlamaCpp


In [ ]:
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

In [ ]:
from langchain_core.prompts import PromptTemplate

# Chains

## Prompt template using a single link chain


In [ ]:
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)


In [ ]:
basic_chain = prompt | llm

In [ ]:
basic_chain.invoke(
    {
        "input_prompt": "Hi! What is 1 + 1?"
    }
)

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


" 1 + 1 equals 2. It's a basic arithmetic operation where you add the number one to another number one, resulting in two."

## Multiple prompt chain

In [ ]:
from langchain_classic.chains import LLMChain

In [ ]:
template = """<s><|user|>Create a title for a story about {summary}. Only return the title.<|end|><|assistant|>"""

title_prompt = PromptTemplate(
    template=template,
    input_variables=["summary"]
)

title = LLMChain(
    llm=llm,
    prompt=title_prompt,
    output_key="title"
)

title.invoke({"summary": "a beautiful day at the lake"})

/tmp/ipykernel_58/2628986360.py:3: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")
/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a beautiful day at the lake',
 'title': ' "Serenity by the Lake: A Day of Blissful Beauty"'}

In [ ]:
template = """<|user|>Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|><|assistant|>"""

character_prompt = PromptTemplate(
    template=template,
    input_variables=["summary", "title"]
)

character = LLMChain(
    llm=llm,
    prompt=character_prompt,
    output_key="character"
)

In [ ]:
template = """<|user|>Create a story about {summary} with the title {title}. The main character is: {character}. Only return the story and it cannot be longer than one paragraph.<|end|><|assistant|>"""

story_prompt = PromptTemplate(
    template=template,
    input_variables=["summary", "title", "character"]
)

story = LLMChain(
    llm=llm,
    prompt=story_prompt,
    output_key="story"
)

In [ ]:
llm_chain = title | character | story

In [ ]:
llm_chain.invoke("a beautiful day on the lake")

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a beautiful day on the lake',
 'title': ' "Luminous Reflections: A Serene Day on the Lake"',
 'character': " The protagonist, a free-spirited artist named Maya, finds solace and inspiration in the tranquil beauty of nature as she spends her day on the serene lake. Her vibrant personality is reflected not only through her lively interactions with fellow boaters but also in the captivating paintings inspired by the shimmering reflections dancing upon the water's surface.",
 'story': " Luminous Reflections: A Serene Day on the Lake\n\nOn a radiant summer morning, Maya, an artist whose spirit danced with every color of nature's palette, set sail on her beloved lake. Her soul swayed in harmony with the gentle waves as she navigated towards her favorite secluded spot—the mirror-like expanse that whispered inspiration to her free-spirited heart. As sunlight kissed the water's surface, Maya marveled at the shimmering reflections that painted a tapestry of iridescent hues across th

# Memory in LLMs

In [ ]:
basic_chain.invoke({"input_prompt": "Hi! My name is Simon. What is 1 + 1?"})

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' Hello Simon! The answer to 1 + 1 is 2.'

In [ ]:
basic_chain.invoke({"input_prompt": "What is my name?"})

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


" I'm unable to determine your name as I don't have access to personal data of individuals. If you need assistance with something specific, feel free to ask!"

In [ ]:
template = """<|user|>Current conversation:{chat_history}
{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

## Conversation Buffer

In [ ]:
from langchain_classic.memory import ConversationBufferMemory

In [ ]:
memory = ConversationBufferMemory(memory_key="chat_history")

/tmp/ipykernel_58/1499110810.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history")


In [ ]:
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [ ]:
llm_chain.invoke({"input_prompt": "Hi! My name is Simon. What is 1 + 1?"})

{'input_prompt': 'Hi! My name is Simon. What is 1 + 1?',
 'chat_history': '',
 'text': " The answer to 1 + 1 is 2.\n\nHowever, it seems like there's a bit more going on in this conversation than just solving the math problem. You might be testing my responsiveness or trying to start a friendly chat! :)"}

In [ ]:
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Simon. What is 1 + 1?\nAI:  The answer to 1 + 1 is 2.\n\nHowever, it seems like there's a bit more going on in this conversation than just solving the math problem. You might be testing my responsiveness or trying to start a friendly chat! :)",
 'text': " Hello Simon! I'm an AI, so I don't have a personal name, but you can call me Assistant. Nice to meet you! :) And your math question is correct: 1 + 1 equals 2. How can I assist you further?"}

## Windowed Conversation Buffer

In [ ]:
from langchain_classic.memory import ConversationBufferWindowMemory

In [ ]:
memory = ConversationBufferWindowMemory(
    k=2,
    memory_key="chat_history"
)

/tmp/ipykernel_58/2660153956.py:1: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(


In [ ]:
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [ ]:
llm_chain.predict(input_prompt="Hi! My name is Simon and I'm 23 years old. What is 1 + 1?")
llm_chain.predict(input_prompt="What is 5 + 5?")

" The answer to 5 + 5 is 10. You're doing well in starting conversations and asking basic questions! If you need help with more complex math problems or have any other queries, feel free to ask. I'm here to assist you!"

In [ ]:
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Simon and I'm 23 years old. What is 1 + 1?\nAI:  The answer to 1 + 1 is 2. It seems like you're also introducing yourself and then posing a simple mathematical question, which I can help with!\n\nHere's a bit more context for the introduction:\n\nHi Simon! It's great to meet you at age 23. You mentioned your name and age first; it's a nice way to start conversations while also sharing some basic information about yourself. As for your question, when we add one plus one, the result is indeed two (1 + 1 = 2). This addition operation is fundamental in mathematics!\n\nIf you have any other questions or need assistance with different topics, feel free to ask. I'm here to help!\nHuman: What is 5 + 5?\nAI:  The answer to 5 + 5 is 10. You're doing well in starting conversations and asking basic questions! If you need help with more complex math problems or have any other queries, feel free to ask. I'm here to assist y

In [ ]:
llm_chain.invoke({"input_prompt":"What is my age?"})

{'input_prompt': 'What is my age?',
 'chat_history': "Human: What is 5 + 5?\nAI:  The answer to 5 + 5 is 10. You're doing well in starting conversations and asking basic questions! If you need help with more complex math problems or have any other queries, feel free to ask. I'm here to assist you!\nHuman: What is my name?\nAI:  Your name is Simon. It's great to continue our conversation! Regarding your new question, 5 + 5 equals 10 (5 + 5 = 10). If there's anything else you'd like to know or discuss, I'm here to help.",
 'text': " I'm unable to access personal data such as age unless you share it with me in the course of our conversation. However, I can help answer questions or provide information about various topics!\nAlternatively, if this is a hypothetical question for learning purposes: Without knowing your birthdate, I cannot determine your actual age. But remember, an accurate way to calculate someone's age would be by subtracting their birth year from the current year. If you h

## Conversation Summary

In [ ]:
summary_prompt_template = """<|user|>Summarize the conversations and update with the new lines.
Current summary:
{summary}

New lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""

summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

In [ ]:
from langchain_classic.memory import ConversationSummaryMemory

In [ ]:
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_58/1333067193.py:1: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(


In [ ]:
llm_chain.invoke({"input_prompt": "Hi! My name is Simon. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': ' Hi! My name is Simon. I asked the AI for the sum of 1 + 1, which was correctly answered as 2 by the AI, followed by an offer to assist further.',
 'text': ' Your name is Simon.'}

In [ ]:
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': " Hi! I'm Simon. Initially, I asked the AI for the sum of 1 + 1, which was correctly answered as 2 by the AI and offered further assistance. Later, when queried about my name, the AI confirmed it to be Simon.",
 'text': ' The first question you asked was "What is 1 + 1?"'}

In [ ]:
memory.load_memory_variables({})

{'chat_history': ' Hi! I\'m Simon. Initially, I asked the AI for the sum of 1 + 1, which was correctly answered as 2 by the AI and offered further assistance. Later, when queried about my name, the AI confirmed it to be Simon. Additionally, upon asking what my first question was, the AI reiterated that I inquired "What is 1 + 1?"'}